In [218]:
import os
import pandas as pd
import numpy as np
from pydbmanager.connection import DatabaseConnection
from pydbmanager.operations import DatabaseOperations
from dotenv import load_dotenv
import warnings
import json
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [219]:
# Load environment variables
load_dotenv()

# Initialize and test database connection
db = DatabaseConnection()
# Initialize database operations (this also creates and stores the connection)
db_ops = DatabaseOperations()

# Check connection status
if db_ops.conn:
    print("\u2705 Connection Successful!")
    db_ops.close()
else:
    print("\u274c Connection Failed!")

2025-05-15 04:14:15,319 - INFO - Database connection established successfully.
2025-05-15 04:14:15,351 - INFO - Database connection closed


✅ Connection Successful!


## 1. Patient Demographics

In [220]:
with open('../../sql/patient_demo.sql', 'r') as file:
    patient_demo_sql = file.read()

In [221]:
patient_demo = db_ops.query_data(patient_demo_sql)
patient_demo.head()

2025-05-15 04:14:19,481 - WARNING - Database connection lost. Reconnecting...
2025-05-15 04:14:19,483 - INFO - Database connection established successfully.
2025-05-15 04:14:19,835 - INFO - Data fetched in 0.3505 seconds


Data fetched successfully!
Dataframe Size (1256, 20)


,patient_id,first_name,last_name,date_of_birth,postal_code,user_type,registered_at,country,referral_group,veteran,ethnicity,race,city,state,dark_mode,gender,patient_type,patient_sub_type,head_hit_count,has_tbi_before
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,CAROLINE,SUONG,1981-12-09,20866,patient,2022-06-08 16:37:57,US,BIA-GA,no,No,Korean,Burtonsville,MD,false,female,TBI/Concussion,tbiPatient,1.0,None
1,00469456-99a5-4c99-aa48-a918986c7c45,Lynsey,Alexander,1981-04-08,G66 8EG,patient,2021-05-19 15:27:06,GB,none,NULL,NULL,NULL,East Dunbartonshire,SCT,false,female,TBI/Concussion,tbiPatient,1.0,None
2,00469456-99a5-4c99-aa48-a918986c7c45,Lynsey,Alexander,1981-04-08,G66 8EG,patient,2021-05-19 21:12:51,GB,none,NULL,NULL,NULL,East Dunbartonshire,SCT,false,female,TBI/Concussion,tbiPatient,1.0,None
3,00e179cd-5edb-47fe-be53-b1e96e905433,Neil,Langrick,1963-11-30,WF178HZ,caregiver,2022-05-28 14:09:05,GB,NONE,notToAnswer,NULL,NULL,West Yorkshire,ENG,false,male,Other,Acquired Brain Injury,NaN,Yes
4,00e179cd-5edb-47fe-be53-b1e96e905433,Neil,Langrick,1963-11-30,WF17 8HZ,patient,2023-04-25 07:13:36,GB,none,no,notToAnswer,Prefer not to answer,zipcodeNotInLibrary,zipcodeNotInLibrary,false,male,Other,Acquired Brain Injury,NaN,Yes


In [222]:
def check_missing_values(df):
    for col in df.columns:
        # Count string-based nulls (case-insensitive)
        string_nulls = df[col].astype(str).str.lower().isin(['null', 'none']).sum()
        
        if string_nulls > 0:
            print(f"Column '{col}' has {string_nulls} missing values (including 'NULL', 'None')")

In [223]:
check_missing_values(patient_demo)

Column 'country' has 220 missing values (including 'NULL', 'None')
Column 'referral_group' has 831 missing values (including 'NULL', 'None')
Column 'veteran' has 706 missing values (including 'NULL', 'None')
Column 'ethnicity' has 717 missing values (including 'NULL', 'None')
Column 'race' has 717 missing values (including 'NULL', 'None')
Column 'city' has 240 missing values (including 'NULL', 'None')
Column 'state' has 240 missing values (including 'NULL', 'None')
Column 'patient_type' has 6 missing values (including 'NULL', 'None')
Column 'patient_sub_type' has 86 missing values (including 'NULL', 'None')
Column 'has_tbi_before' has 1187 missing values (including 'NULL', 'None')


In [224]:
for col in patient_demo.columns:
    patient_demo[col] = (
        patient_demo[col]
        .astype(str)
        .str.strip()  
        .str.lower()
        .replace(['null', 'none', ''], np.nan)
    )

In [225]:
patient_demo['country'].fillna('Not Specified', inplace=True)
patient_demo['city'].fillna('Not Specified', inplace=True)
patient_demo['state'].fillna('Not Specified', inplace=True)

In [226]:
patient_demo['veteran'].fillna('No', inplace=True)
patient_demo['race'].fillna('Not Specified', inplace=True)
patient_demo['referral_group'].fillna('other', inplace=True)
patient_demo['ethnicity'].fillna('No', inplace=True)
patient_demo['patient_type'].fillna('Other', inplace=True)
patient_demo['patient_sub_type'].fillna('Not Specified', inplace=True)

In [227]:
# Convert 'nan' string to actual NaN
patient_demo['head_hit_count'] = patient_demo['head_hit_count'].replace('nan', np.nan)

In [228]:
patient_demo.isna().sum()

patient_id             0
first_name             0
last_name              0
date_of_birth          0
postal_code            0
user_type              0
registered_at          0
country                0
referral_group         0
veteran                0
ethnicity              0
race                   0
city                   0
state                  0
dark_mode              0
gender                 0
patient_type           0
patient_sub_type       0
head_hit_count       292
has_tbi_before      1187
dtype: int64

In [229]:
# patient_demo[patient_demo['head_hit_count'].isna()]

In [230]:
# Now replace NaN with 0
patient_demo['head_hit_count'].fillna(0, inplace=True)

In [231]:
# calculating patient age from date of birth
patient_demo['date_of_birth'] = pd.to_datetime(patient_demo['date_of_birth'])
patient_demo['registered_at'] = pd.to_datetime(patient_demo['registered_at'])
# Calculate age in years    
patient_demo['age'] = (patient_demo['registered_at'] - patient_demo['date_of_birth']).dt.days // 365

In [232]:
patient_demo['has_tbi_before'].fillna('Not Specified', inplace=True)

In [233]:
patient_demo['head_hit_count'] = patient_demo['head_hit_count'].astype('float')

In [234]:
patient_demo.isna().sum()

patient_id          0
first_name          0
last_name           0
date_of_birth       0
postal_code         0
user_type           0
registered_at       0
country             0
referral_group      0
veteran             0
ethnicity           0
race                0
city                0
state               0
dark_mode           0
gender              0
patient_type        0
patient_sub_type    0
head_hit_count      0
has_tbi_before      0
age                 0
dtype: int64

In [235]:
# filtering out the patients with user_type 4
patient_demo['user_type'] = patient_demo['user_type'].str.replace('4','patient')

In [236]:
patient_demo['user_type'].value_counts() 

user_type
patient      1139
caregiver      98
therapist      12
provider        7
Name: count, dtype: int64

In [237]:
# patient_demo.to_clipboard(index=False)

In [238]:
patient_demo.head()

,patient_id,first_name,last_name,date_of_birth,postal_code,user_type,registered_at,country,referral_group,veteran,ethnicity,race,city,state,dark_mode,gender,patient_type,patient_sub_type,head_hit_count,has_tbi_before,age
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,caroline,suong,1981-12-09,20866,patient,2022-06-08 16:37:57,us,bia-ga,no,no,korean,burtonsville,md,false,female,tbi/concussion,tbipatient,1.0,Not Specified,40
1,00469456-99a5-4c99-aa48-a918986c7c45,lynsey,alexander,1981-04-08,g66 8eg,patient,2021-05-19 15:27:06,gb,other,No,No,Not Specified,east dunbartonshire,sct,false,female,tbi/concussion,tbipatient,1.0,Not Specified,40
2,00469456-99a5-4c99-aa48-a918986c7c45,lynsey,alexander,1981-04-08,g66 8eg,patient,2021-05-19 21:12:51,gb,other,No,No,Not Specified,east dunbartonshire,sct,false,female,tbi/concussion,tbipatient,1.0,Not Specified,40
3,00e179cd-5edb-47fe-be53-b1e96e905433,neil,langrick,1963-11-30,wf178hz,caregiver,2022-05-28 14:09:05,gb,other,nottoanswer,No,Not Specified,west yorkshire,eng,false,male,other,acquired brain injury,0.0,yes,58
4,00e179cd-5edb-47fe-be53-b1e96e905433,neil,langrick,1963-11-30,wf17 8hz,patient,2023-04-25 07:13:36,gb,other,no,nottoanswer,prefer not to answer,zipcodenotinlibrary,zipcodenotinlibrary,false,male,other,acquired brain injury,0.0,yes,59


In [239]:
patient_demo = patient_demo[~patient_demo['patient_id'].duplicated()]

In [310]:
patient_demo.shape

(1107, 21)

In [ ]:
patient_demo.dtypes

patient_id                  object
first_name                  object
last_name                   object
date_of_birth       datetime64[ns]
postal_code                 object
user_type                   object
registered_at       datetime64[ns]
country                     object
referral_group              object
veteran                     object
ethnicity                   object
race                        object
city                        object
state                       object
dark_mode                   object
gender                      object
patient_type                object
patient_sub_type            object
head_hit_count             float64
has_tbi_before              object
age                          int64
dtype: object

In [241]:
# db_ops.insert_dataframe(patient_demo, table_name='patient_demographics')

In [242]:
# patient_demo.to_csv('../../data/clean/patient_demo.csv', index=False)

## 2. Incident Registration data

In [243]:
with open('../../sql/tbi_incident_registration_data.sql', 'r') as file:
    tbi_incident_reg_sql = file.read()

In [244]:
tbi_incident_reg = db_ops.query_data(tbi_incident_reg_sql)
tbi_incident_reg.head()

2025-05-15 04:18:42,297 - INFO - Data fetched in 0.0446 seconds


Data fetched successfully!
Dataframe Size (954, 8)


,patient_id,tbi_incident_date,injury_from,head_hit_location,num_head_hit_location,total_tbi,immediate_symptoms_resulting,describe_event
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-02,Accident,Top of Head,1,1,"Confusion, Dazed or Vacant Stare, Disorientati...","Automobile accident, ran into cars and flipped..."
1,00469456-99a5-4c99-aa48-a918986c7c45,2021-02-09,Subarachnoid haemorrhage,Right Side of Head,1,1,"Confusion, Dazed or Vacant Stare, Disorientati...",I collapsed at work showing stroke like sympto...
2,012bfc75-70c1-4007-ab9e-1f4ee45bd537,2014-08-12,Accident,Entire head. texture trailer from semi that hi...,1,2,"Loss of Consciousness,Disorientation,Incoheren...",Hit by a car checking my mail. The car hit me ...
3,012eccde-f0b9-4ae2-aebf-3c821c461545,2022-10-21,Fall,Left Side of Head,1,1,"Incoherent Speech,Disorientation,Confusion,Daz...",I fell down stairs
4,01786d46-2829-42e2-8d50-135b7e212ea3,2019-03-01,Accident,Back of Head,1,3,"Disorientation,Confusion,Extreme pain in the f...",Not sure which day in March. I went to sit bac...


In [245]:
tbi_incident_reg.head(20).to_clipboard()

In [8]:
# tbi_incident_reg.to_clipboard()

In [246]:
tbi_incident_reg.isna().sum()

patient_id                       0
tbi_incident_date                0
injury_from                      7
head_hit_location               21
num_head_hit_location            0
total_tbi                        0
immediate_symptoms_resulting    10
describe_event                   0
dtype: int64

In [247]:
## total tbi we will cap at 30
## num head hit location will put zero for na values
## injury_from we will put not known for na values
## immediate_symptioms_resulting we will put None for na values

In [248]:
tbi_incident_reg.dtypes

patient_id                      object
tbi_incident_date               object
injury_from                     object
head_hit_location               object
num_head_hit_location           object
total_tbi                        int64
immediate_symptoms_resulting    object
describe_event                  object
dtype: object

In [249]:
tbi_incident_reg['total_tbi'] = np.where(tbi_incident_reg['total_tbi'] <= 30, tbi_incident_reg['total_tbi'], 30)
tbi_incident_reg['num_head_hit_location'] = tbi_incident_reg['num_head_hit_location'].str.lower().replace('null', 0)
tbi_incident_reg['num_head_hit_location'] = tbi_incident_reg['num_head_hit_location'].astype(int).fillna(0).astype(int)
tbi_incident_reg['injury_from'] = tbi_incident_reg['injury_from'].fillna('Not Known')
tbi_incident_reg['immediate_symptoms_resulting'] = tbi_incident_reg['immediate_symptoms_resulting'].fillna('None')
tbi_incident_reg['head_hit_location'] = tbi_incident_reg['head_hit_location'].fillna('Not known')

In [250]:
tbi_incident_reg['tbi_incident_date'] = pd.to_datetime(tbi_incident_reg['tbi_incident_date'],errors='coerce')

In [251]:
max(tbi_incident_reg['tbi_incident_date']), min(tbi_incident_reg['tbi_incident_date'])

(Timestamp('2025-01-29 00:00:00'), Timestamp('1901-01-01 00:00:00'))

In [252]:
tbi_incident_reg.isna().sum()

patient_id                      0
tbi_incident_date               1
injury_from                     0
head_hit_location               0
num_head_hit_location           0
total_tbi                       0
immediate_symptoms_resulting    0
describe_event                  0
dtype: int64

In [253]:
tbi_incident_reg.shape

(954, 8)

In [254]:
tbi_incident_reg['patient_id'].nunique()

954

In [255]:
tbi_incident_reg.head(2)

,patient_id,tbi_incident_date,injury_from,head_hit_location,num_head_hit_location,total_tbi,immediate_symptoms_resulting,describe_event
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-02,Accident,Top of Head,1,1,"Confusion, Dazed or Vacant Stare, Disorientati...","Automobile accident, ran into cars and flipped..."
1,00469456-99a5-4c99-aa48-a918986c7c45,2021-02-09,Subarachnoid haemorrhage,Right Side of Head,1,1,"Confusion, Dazed or Vacant Stare, Disorientati...",I collapsed at work showing stroke like sympto...


In [256]:
tbi_incident_reg.dtypes

patient_id                              object
tbi_incident_date               datetime64[ns]
injury_from                             object
head_hit_location                       object
num_head_hit_location                    int64
total_tbi                                int64
immediate_symptoms_resulting            object
describe_event                          object
dtype: object

In [257]:
# tbi_incident_reg.to_csv('../../data/clean/tbi_incident_register.csv', index=False)

## 3. Worst Top 3 symptoms

In [258]:
with open('../../sql/worst_3_symptoms.sql') as file:
    worst_3_symptoms_sql = file.read()

In [259]:
worst_symptoms = db_ops.query_data(worst_3_symptoms_sql)
worst_symptoms.head()

2025-05-15 04:20:10,821 - INFO - Data fetched in 0.1083 seconds


Data fetched successfully!
Dataframe Size (1192, 6)


,patient_id,symptom_id,id,category,subcategory,factor
0,00e179cd-5edb-47fe-be53-b1e96e905433,7,7,medical,cognitive,"Brain Fog, Lack of Focus"
1,00e179cd-5edb-47fe-be53-b1e96e905433,8,8,medical,cognitive,Short Term Memory Loss
2,00e179cd-5edb-47fe-be53-b1e96e905433,10,10,medical,cognitive,Slow Thinking or Processing
3,012eccde-f0b9-4ae2-aebf-3c821c461545,7,7,medical,cognitive,"Brain Fog, Lack of Focus"
4,012eccde-f0b9-4ae2-aebf-3c821c461545,8,8,medical,cognitive,Short Term Memory Loss


In [260]:
worst_symptoms.shape

(1192, 6)

In [261]:
worst_symptoms.isna().sum()

patient_id     0
symptom_id     0
id             0
category       0
subcategory    0
factor         0
dtype: int64

In [262]:
def build_json(group):
    category = group['category'].iloc[0]  
    worst_symptoms = group[['subcategory', 'factor']].to_dict(orient='records')
    return {'category': category, 'worst symptoms': worst_symptoms}

In [263]:
worst_3_symptoms = worst_symptoms.groupby('patient_id').apply(build_json).reset_index(name='worst_symptoms')

In [264]:
worst_3_symptoms.dtypes

patient_id        object
worst_symptoms    object
dtype: object

In [266]:
# worst_3_symptoms.head(2)
worst_3_symptoms.shape

(426, 2)

In [ ]:
# worst_symptoms.to_csv('../../data/clean/top_3_worst_symptoms.csv', index=False)

In [17]:
# worst_symptoms.to_clipboard()

## 4. ALL RECORDED MEDICAL SYMPTOMS

In [267]:
with open('../../sql/recorded_medical_symptoms.sql') as file:
    medical_symptoms_sql = file.read()

In [268]:
med_sympt = db_ops.query_data(medical_symptoms_sql)
med_sympt.head()

2025-05-15 04:23:58,764 - INFO - Data fetched in 0.4560 seconds


Data fetched successfully!
Dataframe Size (6986, 7)


,patient_id,symptom_id,id,category,subcategory,factor,prime
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,26,26,medical,physical,Lack of Coordination,true
1,0006ad41-c2d3-4994-8aab-7a3a107d50aa,511,511,medical,speech,Limited social engagement,true
2,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1009,1009,medical,cognitive,Can't sleep or relax,NULL
3,00469456-99a5-4c99-aa48-a918986c7c45,3,3,medical,sleep,Fatigue,true
4,00469456-99a5-4c99-aa48-a918986c7c45,8,8,medical,cognitive,Short Term Memory Loss,true


In [269]:
med_sympt.shape

(6986, 7)

In [270]:
med_sympt.isna().sum()

patient_id     0
symptom_id     0
id             0
category       0
subcategory    0
factor         0
prime          0
dtype: int64

In [271]:
# med_sympt.to_csv('../../data/clean/all_recorded_medical_symptoms.csv', index=False)

In [272]:
def build_json(group):
    category = group['category'].iloc[0]  
    symptoms = group[['subcategory', 'factor', 'prime']].to_dict(orient='records')
    return {'category': category, 'symptoms': symptoms}

In [273]:
med_sympt_result = med_sympt.groupby('patient_id').apply(build_json).reset_index(name='symptom_json')

In [274]:
med_sympt_result.shape

(983, 2)

In [276]:
med_sympt_result['patient_id'].nunique()

983

In [275]:
med_sympt_result.head()

,patient_id,symptom_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"{'category': 'medical', 'symptoms': [{'subcate..."
1,00469456-99a5-4c99-aa48-a918986c7c45,"{'category': 'medical', 'symptoms': [{'subcate..."
2,00e179cd-5edb-47fe-be53-b1e96e905433,"{'category': 'medical', 'symptoms': [{'subcate..."
3,012bfc75-70c1-4007-ab9e-1f4ee45bd537,"{'category': 'medical', 'symptoms': [{'subcate..."
4,012eccde-f0b9-4ae2-aebf-3c821c461545,"{'category': 'medical', 'symptoms': [{'subcate..."


In [147]:
# med_sympt_result.to_clipboard()

## 5. ALL SDOH RECORDS  AT THE TIME OF REGISTRATION

In [277]:
with open('../../sql/SDOH_data.sql') as file:
    sdoh_sql = file.read()

In [278]:
sdoh_df = db_ops.query_data(sdoh_sql)
sdoh_df.head()

2025-05-15 04:34:26,681 - INFO - Data fetched in 0.1777 seconds


Data fetched successfully!
Dataframe Size (5219, 5)


,patient_id,symptom_id,category,subcategory,factor
0,00e179cd-5edb-47fe-be53-b1e96e905433,52,SDOH,wellness,Exercise
1,00e179cd-5edb-47fe-be53-b1e96e905433,64,SDOH,wellness,Stress
2,00e179cd-5edb-47fe-be53-b1e96e905433,1136,SDOH,wellness,Muscle Pain
3,012eccde-f0b9-4ae2-aebf-3c821c461545,32,SDOH,travel,Car
4,012eccde-f0b9-4ae2-aebf-3c821c461545,50,SDOH,wellness,Dehydration


In [279]:
sdoh_df.shape

(5219, 5)

In [280]:
# sdoh_df.to_csv('../../data/clean/sdoh_data.csv', index=False)

In [281]:
# sdoh_df.to_clipboard()

In [282]:
def build_sdoh_json(group):
    category = group['category'].iloc[0]
    subcat_dict = (
        group.groupby('subcategory')['factor']
        .apply(lambda x: sorted(set(x.dropna())))
        .to_dict()
    )
    return {'category': category, 'subcategories': subcat_dict}

In [283]:
sdoh_df_result = sdoh_df.groupby('patient_id').apply(build_sdoh_json).reset_index(name='sdoh_json')

In [284]:
sdoh_df_result.shape

(409, 2)

In [285]:
sdoh_df_result.head()

,patient_id,sdoh_json
0,00e179cd-5edb-47fe-be53-b1e96e905433,"{'category': 'SDOH', 'subcategories': {'wellne..."
1,012eccde-f0b9-4ae2-aebf-3c821c461545,"{'category': 'SDOH', 'subcategories': {'travel..."
2,021a5e27-8029-4f0c-b268-cee04dbc40b9,"{'category': 'SDOH', 'subcategories': {'Dietar..."
3,022242f1-bcae-4230-9b5d-e03b1e3e25fa,"{'category': 'SDOH', 'subcategories': {'wellne..."
4,02b72dc4-3645-4f19-be5a-989e9147544e,"{'category': 'SDOH', 'subcategories': {'enviro..."


In [286]:
# sdoh_df_result.to_clipboard()

## 6. ALL THERAPIES AT THE TIME OF REGISTRATION

In [287]:
with open('../../sql/therapies_at_reg.sql') as file:
    therapies_sql = file.read() 

In [288]:
therap_df = db_ops.query_data(therapies_sql)
therap_df.head()

2025-05-15 04:35:04,313 - INFO - Data fetched in 0.0569 seconds


Data fetched successfully!
Dataframe Size (1866, 4)


,patient_id,therapies_id,therapies,category
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,11,Acupuncture,NULL
1,0006ad41-c2d3-4994-8aab-7a3a107d50aa,12,Chiropractic / Functional Neurology,NULL
2,0006ad41-c2d3-4994-8aab-7a3a107d50aa,13,Massage Therapy,NULL
3,0006ad41-c2d3-4994-8aab-7a3a107d50aa,14,Pain Management,NULL
4,0006ad41-c2d3-4994-8aab-7a3a107d50aa,15,Physical Therapy,NULL


In [289]:
therap_df.isna().sum()

patient_id      0
therapies_id    0
therapies       0
category        0
dtype: int64

In [290]:
# therap_df.to_csv('../../data/clean/therapies_at_registration.csv', index=False)

In [291]:
# therap_df.to_clipboard()

In [292]:
def build_therapy_json(group):
    therapy_dict = (
        group.groupby('category')['therapies']
        .apply(lambda x: sorted(set(x.dropna())))
        .to_dict()
    )
    return therapy_dict

In [293]:
therap_df_result = therap_df.groupby('patient_id').apply(build_therapy_json).reset_index(name='therapy_json')
therap_df_result.shape

(399, 2)

In [294]:
therap_df_result.head()

,patient_id,therapy_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"{'NULL': ['Acupuncture', 'Chiropractic / Funct..."
1,00e179cd-5edb-47fe-be53-b1e96e905433,{'NULL': ['None']}
2,021a5e27-8029-4f0c-b268-cee04dbc40b9,"{'Alternative': ['Pain Management'], 'NULL': [..."
3,022242f1-bcae-4230-9b5d-e03b1e3e25fa,"{'Applied': ['Physical Therapy'], 'Chiropracti..."
4,02b72dc4-3645-4f19-be5a-989e9147544e,"{'Applied': ['Occupational Therapy', 'Physical..."


In [171]:
# therap_df_result.to_clipboard()

## 7. SYMPTOM LOGS OVER TIME

In [295]:
with open('../../sql/symptom_logs.sql') as file:
    symptom_logs_sql = file.read()  

In [296]:
symptom_logs = db_ops.query_data(symptom_logs_sql)
symptom_logs.head()

2025-05-15 04:35:59,123 - INFO - Data fetched in 7.2781 seconds


Data fetched successfully!
Dataframe Size (187218, 8)


,patient_about_id,symptom_date,logged_at,severity,category,subcategory,had_symptom,factor
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.665091,NULL,medical,cognitive,true,"Brain Fog, Lack of Focus"
1,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,emotional,true,Anxiety
2,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,emotional,true,Depression
3,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,emotional,true,No Motivation
4,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,sleep,true,Constipation


In [297]:
symptom_logs.isna().sum()

patient_about_id    0
symptom_date        0
logged_at           0
severity            0
category            0
subcategory         0
had_symptom         0
factor              4
dtype: int64

In [298]:
symptom_logs['factor'].fillna('Not Specified', inplace=True)

In [299]:
symptom_logs.isna().sum()

patient_about_id    0
symptom_date        0
logged_at           0
severity            0
category            0
subcategory         0
had_symptom         0
factor              0
dtype: int64

In [300]:
symptom_logs['symptom_date'] = pd.to_datetime(symptom_logs['symptom_date'], errors='coerce')
symptom_logs['logged_at'] = pd.to_datetime(symptom_logs['logged_at'], errors='coerce')
symptom_logs['logged_at'].max(), symptom_logs['logged_at'].min()

(Timestamp('2025-03-17 13:36:00.316902'),
 Timestamp('2021-03-20 19:42:17.427278'))

In [301]:
symptom_logs.dropna(subset=['symptom_date'], inplace=True)

In [302]:
symptom_logs.dropna(subset=['logged_at'], inplace=True)

In [303]:
symptom_logs.shape

(153394, 8)

In [304]:
symptom_logs.isna().sum()

patient_about_id    0
symptom_date        0
logged_at           0
severity            0
category            0
subcategory         0
had_symptom         0
factor              0
dtype: int64

In [305]:
symptom_logs['severity'] = symptom_logs['severity'].replace(['NULL', 'None', 'Not Specified'], np.nan)
symptom_logs['severity'] = symptom_logs['severity'].astype(float)

In [306]:
symptom_logs['severity'].fillna(0, inplace=True)

In [307]:
symptom_logs.isna().sum()

patient_about_id    0
symptom_date        0
logged_at           0
severity            0
category            0
subcategory         0
had_symptom         0
factor              0
dtype: int64

In [308]:
symptom_logs.dtypes

patient_about_id            object
symptom_date        datetime64[ns]
logged_at           datetime64[ns]
severity                   float64
category                    object
subcategory                 object
had_symptom                 object
factor                      object
dtype: object

In [309]:
# symptom_logs.to_csv('../../data/clean/symptom_logs.csv', index=False)
# # symptom_logs.to_clipboard()

## Joining all the major tables 

In [311]:
# Start with base table: patient_demo
summary_df = patient_demo.copy()

In [312]:
# Define all tables to merge (on 'patient_id')
tables_to_merge = [
    tbi_incident_reg,
    worst_3_symptoms,
    med_sympt_result,
    sdoh_df_result,
    therap_df_result
]

# Merge each table onto summary_df using left join on 'patient_id'
for table in tables_to_merge:
    summary_df = summary_df.merge(table, on='patient_id', how='left')

In [313]:
# Display final shape and preview
print(f"Final summary_df shape: {summary_df.shape}")
summary_df.head()

Final summary_df shape: (1107, 32)


,patient_id,first_name,last_name,date_of_birth,postal_code,user_type,registered_at,country,referral_group,veteran,ethnicity,race,city,state,dark_mode,gender,patient_type,patient_sub_type,head_hit_count,has_tbi_before,age,tbi_incident_date,injury_from,head_hit_location,num_head_hit_location,total_tbi,immediate_symptoms_resulting,describe_event,worst_symptoms,symptom_json,sdoh_json,therapy_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,caroline,suong,1981-12-09,20866,patient,2022-06-08 16:37:57,us,bia-ga,no,no,korean,burtonsville,md,false,female,tbi/concussion,tbipatient,1.0,Not Specified,40,1999-07-02,Accident,Top of Head,1.0,1.0,"Confusion, Dazed or Vacant Stare, Disorientati...","Automobile accident, ran into cars and flipped...",NaN,"{'category': 'medical', 'symptoms': [{'subcate...",NaN,"{'NULL': ['Acupuncture', 'Chiropractic / Funct..."
1,00469456-99a5-4c99-aa48-a918986c7c45,lynsey,alexander,1981-04-08,g66 8eg,patient,2021-05-19 15:27:06,gb,other,No,No,Not Specified,east dunbartonshire,sct,false,female,tbi/concussion,tbipatient,1.0,Not Specified,40,2021-02-09,Subarachnoid haemorrhage,Right Side of Head,1.0,1.0,"Confusion, Dazed or Vacant Stare, Disorientati...",I collapsed at work showing stroke like sympto...,NaN,"{'category': 'medical', 'symptoms': [{'subcate...",NaN,NaN
2,00e179cd-5edb-47fe-be53-b1e96e905433,neil,langrick,1963-11-30,wf178hz,caregiver,2022-05-28 14:09:05,gb,other,nottoanswer,No,Not Specified,west yorkshire,eng,false,male,other,acquired brain injury,0.0,yes,58,NaT,NaN,NaN,NaN,NaN,NaN,NaN,"{'category': 'medical', 'worst symptoms': [{'s...","{'category': 'medical', 'symptoms': [{'subcate...","{'category': 'SDOH', 'subcategories': {'wellne...",{'NULL': ['None']}
3,012bfc75-70c1-4007-ab9e-1f4ee45bd537,june,robison,2001-06-01,32958,patient,2020-08-31 01:57:58,Not Specified,other,No,No,Not Specified,Not Specified,Not Specified,false,female,tbi/concussion,tbipatient,1.0,Not Specified,19,2014-08-12,Accident,Entire head. texture trailer from semi that hi...,1.0,2.0,"Loss of Consciousness,Disorientation,Incoheren...",Hit by a car checking my mail. The car hit me ...,NaN,"{'category': 'medical', 'symptoms': [{'subcate...",NaN,NaN
4,012eccde-f0b9-4ae2-aebf-3c821c461545,betsy,plonski,1957-07-28,21045,patient,2023-08-07 17:29:25,us,other,no,otherethnicities,white,columbia,md,false,female,tbi/concussion,post-concussion symptom (pcs),4.0,Not Specified,66,2022-10-21,Fall,Left Side of Head,1.0,1.0,"Incoherent Speech,Disorientation,Confusion,Daz...",I fell down stairs,"{'category': 'medical', 'worst symptoms': [{'s...","{'category': 'medical', 'symptoms': [{'subcate...","{'category': 'SDOH', 'subcategories': {'travel...",NaN


In [314]:
summary_df.dtypes

patient_id                              object
first_name                              object
last_name                               object
date_of_birth                   datetime64[ns]
postal_code                             object
user_type                               object
registered_at                   datetime64[ns]
country                                 object
referral_group                          object
veteran                                 object
ethnicity                               object
race                                    object
city                                    object
state                                   object
dark_mode                               object
gender                                  object
patient_type                            object
patient_sub_type                        object
head_hit_count                         float64
has_tbi_before                          object
age                                      int64
tbi_incident_

In [ ]:
# summary_df.to_csv('../../data/clean/patients_summary.csv', index=False)

In [316]:
# summary_df.isna().sum()

### Testing Updated PyDBManager package

In [45]:
therap_df_result.shape, therap_df_result.dtypes

((399, 2),
 patient_id      object
 therapy_json    object
 dtype: object)

In [46]:
therap_df_result.head()

,patient_id,therapy_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"{'NULL': ['Acupuncture', 'Chiropractic / Funct..."
1,00e179cd-5edb-47fe-be53-b1e96e905433,{'NULL': ['None']}
2,021a5e27-8029-4f0c-b268-cee04dbc40b9,"{'Alternative': ['Pain Management'], 'NULL': [..."
3,022242f1-bcae-4230-9b5d-e03b1e3e25fa,"{'Applied': ['Physical Therapy'], 'Chiropracti..."
4,02b72dc4-3645-4f19-be5a-989e9147544e,"{'Applied': ['Occupational Therapy', 'Physical..."


In [48]:
create_table_sql = """
IF NOT EXISTS (
    SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'patient_symptoms'
)
BEGIN
    CREATE TABLE patient_symptoms (
        patient_id VARCHAR(100) PRIMARY KEY,
        therapy_json NVARCHAR(MAX)
    );
END
"""
db_ops.create_table(create_table_sql)

2025-05-11 04:18:06,112 - INFO - Query executed successfully!


True

In [50]:
therap_df_result['therapy_json'] = therap_df_result['therapy_json'].apply(lambda x: json.dumps(x))

# Now insert
success = db_ops.insert_dataframe(therap_df_result, table_name='patient_symptoms')


2025-05-11 04:18:38,666 - INFO - Inserted DataFrame into table 'patient_symptoms' successfully!
